# Data Preprocessing & Feature Engineering

## Project
AI Recruitment Guardian: An Explainable Machine Learning Framework for Fake Job Detection and Risk Assessment

### Objectives
- Handle missing values per the recommendation table from `01_Data_Understanding.ipynb`.
- Clean and normalize text fields.
- Engineer metadata and suspicious-content signal features.
- Vectorize text with TF-IDF.
- Encode categorical features.
- Produce a stratified train/test split.

This notebook picks up exactly where the EDA notebook left off. The missing-value
treatment plan below follows Part 3.5 of `01_Data_Understanding.ipynb` — no new
assumptions are introduced here, it is implemented as planned.


In [1]:
import pandas as pd
import numpy as np
import re
import string
import pickle
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix

pd.set_option("display.max_columns", None)
print("Libraries imported successfully! ✅")

Libraries imported successfully! ✅


## Part A: Load Data

Same source file used in the EDA notebook.

In [3]:
df = pd.read_csv("../data/raw/fake_job_postings.csv", engine="python", on_bad_lines="skip")
print("Shape:", df.shape)
df.head()

Shape: (17880, 18)


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


## Part B: Handle Missing Values

Following the recommendation table from the EDA notebook:

| Feature | Treatment |
|---|---|
| `salary_range` | Drop as numeric; keep only a binary `has_salary` flag (84% missing, unreliable format) |
| `department` | Drop; keep only a binary `has_department` flag (65% missing, high-cardinality free text) |
| `required_experience`, `required_education`, `function`, `employment_type`, `industry` | Impute with `"Not Specified"` — missingness itself may be predictive of fraud, so rows are kept, not dropped |
| `company_profile`, `benefits`, `requirements` | Impute with empty string — these remain usable as text features even when blank |
| `location` | Impute with `"Unknown"` |
| `title`, `description` | Already near-fully populated; imputed defensively with empty string in case of edge cases |

**Why flags instead of dropping rows:** dropping every row with a missing value
would remove the vast majority of the dataset (`salary_range` alone is missing
84% of the time). Converting high-missingness columns into a binary "was this
field filled in?" signal preserves the row while still letting the model learn
from the *absence* of information — which fraud-detection literature consistently
finds informative (scammers omit verifiable details).

In [4]:
df["has_salary"] = df["salary_range"].notnull().astype(int)
df["has_department"] = df["department"].notnull().astype(int)
df = df.drop(columns=["salary_range", "department"])

categorical_impute_cols = ["required_experience", "required_education",
                            "function", "employment_type", "industry"]
for col in categorical_impute_cols:
    df[col] = df[col].fillna("Not Specified")

text_impute_cols = ["company_profile", "benefits", "requirements", "title", "description"]
for col in text_impute_cols:
    df[col] = df[col].fillna("")

df["location"] = df["location"].fillna("Unknown")

print("Remaining missing values:", df.isnull().sum().sum())
assert df.isnull().sum().sum() == 0, "Missing values remain — check column list above"
df.info()

Remaining missing values: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   job_id               17880 non-null  int64 
 1   title                17880 non-null  object
 2   location             17880 non-null  object
 3   company_profile      17880 non-null  object
 4   description          17880 non-null  object
 5   requirements         17880 non-null  object
 6   benefits             17880 non-null  object
 7   telecommuting        17880 non-null  int64 
 8   has_company_logo     17880 non-null  int64 
 9   has_questions        17880 non-null  int64 
 10  employment_type      17880 non-null  object
 11  required_experience  17880 non-null  object
 12  required_education   17880 non-null  object
 13  industry             17880 non-null  object
 14  function             17880 non-null  object
 15  fraudulent           1788

## Part C: Text Cleaning

Each free-text column is cleaned independently before being combined into a
single `full_text` field for vectorization. Steps, in order:

1. **Lowercase** — so "Fee" and "fee" are treated as the same token.
2. **Strip HTML tags** — many EMSCAD descriptions were originally scraped from web postings and contain leftover `<p>`, `<br>` etc.
3. **Strip URLs and emails** — removed from the text used for TF-IDF (raw counts of them are captured separately as *metadata features* in Part D, since "does this posting contain a Gmail address" is a stronger fraud signal as a yes/no flag than as vocabulary).
4. **Strip punctuation** — reduces noise/sparsity in the TF-IDF vocabulary.
5. **Strip standalone digits** — salary figures and phone numbers create thousands of near-unique numeric tokens that just add noise to a bag-of-words model.
6. **Collapse whitespace**.

We deliberately do **not** stem/lemmatize here — TF-IDF with `ngram_range=(1,2)`
on raw cleaned tokens keeps phrase-level signals like "registration fee" intact,
which matters for the Explainable AI step later (SHAP explanations are more
human-readable on real words than on stemmed fragments).

In [5]:
SUSPICIOUS_PHRASES = [
    "registration fee", "no experience required", "immediate joining",
    "no interview", "limited seats", "earn money fast", "work from home",
    "quick money", "easy money", "wire transfer", "processing fee"
]

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)              # strip HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # strip URLs
    text = re.sub(r"\S+@\S+", " ", text)             # strip emails
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)                 # strip standalone numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

for col in ["title", "company_profile", "description", "requirements", "benefits"]:
    df[col + "_clean"] = df[col].apply(clean_text)

df["full_text"] = (
    df["title_clean"] + " " + df["company_profile_clean"] + " " +
    df["description_clean"] + " " + df["requirements_clean"] + " " +
    df["benefits_clean"]
)

df[["title", "title_clean"]].head(3)

,title,title_clean
0,Marketing Intern,marketing intern
1,Customer Service - Cloud Video Production,customer service cloud video production
2,Commissioning Machinery Assistant (CMA),commissioning machinery assistant cma


## Part D: Feature Engineering

Two families of engineered features, beyond the text itself:

**1. Suspicious phrase flags** — one binary column per phrase in `SUSPICIOUS_PHRASES`,
searched against the *raw* (uncleaned) text so exact phrases like "registration
fee" are matched reliably. These feed directly into the Job Trust Score and the
"Highlighted Suspicious Phrases" dashboard feature later — this is the same list
that module will reuse.

**2. Metadata-derived signals**:
- `suspicious_phrase_count` — sum of all flags, a simple aggregate risk signal.
- `description_length` — word count of the cleaned description; genuine postings
  tend to be longer and more detailed than scam postings, which are often short
  and generic.
- `has_email_gmail` — whether the description contains a Gmail address; legitimate
  companies overwhelmingly post from a corporate domain, not a free webmail
  address, so this is one of the strongest individual red flags in fraud-detection
  literature on job postings.

These, plus the already-existing `telecommuting`, `has_company_logo`,
`has_questions`, `has_salary`, `has_department` columns, form the **metadata
feature block** that gets concatenated with the TF-IDF text block in Part F.

In [6]:
raw_combined_lower = (
    df["title"] + " " + df["description"] + " " + df["requirements"]
).str.lower()

for phrase in SUSPICIOUS_PHRASES:
    col_name = "flag_" + re.sub(r"\W+", "_", phrase)
    df[col_name] = raw_combined_lower.str.contains(phrase, regex=False).astype(int)

flag_cols = [c for c in df.columns if c.startswith("flag_")]
df["suspicious_phrase_count"] = df[flag_cols].sum(axis=1)
df["description_length"] = df["description_clean"].apply(lambda x: len(x.split()))
df["has_email_gmail"] = df["description"].str.contains("gmail.com", case=False, na=False).astype(int)

print("Engineered flag columns:", flag_cols)
df[flag_cols + ["suspicious_phrase_count", "description_length", "has_email_gmail"]].head()

Engineered flag columns: ['flag_registration_fee', 'flag_no_experience_required', 'flag_immediate_joining', 'flag_no_interview', 'flag_limited_seats', 'flag_earn_money_fast', 'flag_work_from_home', 'flag_quick_money', 'flag_easy_money', 'flag_wire_transfer', 'flag_processing_fee']


,flag_registration_fee,flag_no_experience_required,flag_immediate_joining,flag_no_interview,flag_limited_seats,flag_earn_money_fast,flag_work_from_home,flag_quick_money,flag_easy_money,flag_wire_transfer,flag_processing_fee,suspicious_phrase_count,description_length,has_email_gmail
0,0,0,0,0,0,0,0,0,0,0,0,0,124,0
1,0,0,0,0,0,0,0,0,0,0,0,0,300,0
2,0,0,0,0,0,0,0,0,0,0,0,0,50,0
3,0,0,0,0,0,0,0,0,0,0,0,0,346,0
4,0,0,0,0,0,0,0,0,0,0,0,0,168,0


## Part E: TF-IDF Vectorization

Converts `full_text` into a sparse numeric matrix.

- `max_features=3000` — caps the vocabulary. This is deliberate: an uncapped
  TF-IDF vocabulary can run into tens of thousands of columns, which makes later
  SHAP explanations slow and unreadable (nobody wants a waterfall plot with 20,000
  bars). 3000 keeps the informative vocabulary while staying interpretable.
- `ngram_range=(1, 2)` — includes both single words and two-word phrases, so
  bigrams like "registration fee" or "no experience" are captured as single
  features, not lost across two separate unigrams.
- `min_df=2` — drops terms that appear in only one document (near-certainly noise/typos).
- `max_df=0.9` — drops terms that appear in over 90% of documents (too common to be discriminative).
- `stop_words="english"` — removes common English stopwords (the, is, and, ...).

The fitted vectorizer is saved to disk in Part H so the **same** vocabulary and
IDF weights are reused at inference time in the Streamlit app — refitting on new
input text would silently break the model.

In [7]:
tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    stop_words="english"
)
X_text = tfidf.fit_transform(df["full_text"])
print("TF-IDF matrix shape:", X_text.shape)

TF-IDF matrix shape: (17880, 3000)


## Part F: Categorical Encoding

One-hot encoding for the remaining categorical columns
(`employment_type`, `required_experience`, `required_education`, `industry`,
`function`). `handle_unknown="ignore"` is important here: at inference time in
the Streamlit app, a user-entered category the encoder never saw during
training (e.g. a rare industry) will be encoded as all-zeros instead of
crashing the pipeline.

In [8]:
categorical_cols = ["employment_type", "required_experience",
                     "required_education", "industry", "function"]

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_cat = ohe.fit_transform(df[categorical_cols])
print("Categorical encoded shape:", X_cat.shape)
print("Example categories learned for 'employment_type':",
      ohe.categories_[categorical_cols.index("employment_type")])

Categorical encoded shape: (17880, 198)
Example categories learned for 'employment_type': ['Contract' 'Full-time' 'Not Specified' 'Other' 'Part-time' 'Temporary']


## Part G: Assemble the Final Feature Matrix

Text (TF-IDF) + categorical (one-hot) + numeric/binary metadata are combined
into a **single sparse matrix** using `scipy.sparse.hstack`. This is the matrix
every model in the comparison step (Logistic Regression, Naive Bayes, SVM,
Random Forest, XGBoost) will be trained on — all models see the exact same
feature space, which is what makes the model comparison fair.

In [9]:
numeric_cols = (
    ["telecommuting", "has_company_logo", "has_questions", "has_salary",
     "has_department", "suspicious_phrase_count", "description_length",
     "has_email_gmail"]
    + flag_cols
)
X_numeric = csr_matrix(df[numeric_cols].values.astype(float))

X = hstack([X_text, X_cat, X_numeric]).tocsr()
y = df["fraudulent"].values

print("TF-IDF block:      ", X_text.shape)
print("Categorical block: ", X_cat.shape)
print("Numeric block:     ", X_numeric.shape)
print("Final combined X:  ", X.shape)
print("Label vector y:    ", y.shape)

TF-IDF block:       (17880, 3000)
Categorical block:  (17880, 198)
Numeric block:      (17880, 19)
Final combined X:   (17880, 3217)
Label vector y:     (17880,)


## Part H: Train/Test Split (Stratified)

The EMSCAD dataset is heavily **imbalanced** — fraudulent postings are a small
minority (roughly 5% in the full dataset). A plain random split can easily
produce a test set with an even smaller (or larger) fraud rate than the train
set, which distorts every downstream metric.

`stratify=y` forces the class ratio in `y_train` and `y_test` to match the
overall dataset ratio. This one parameter is also why **accuracy** will not be
used as the primary evaluation metric later — a model that predicts "real" for
every posting would already score ~95% accuracy while being useless. Precision,
recall, F1, and ROC-AUC (already planned in the model comparison step) are what
actually matter here.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train fraud rate: %.2f%%" % (y_train.mean() * 100))
print("Test fraud rate:  %.2f%%" % (y_test.mean() * 100))

Train shape: (14304, 3217)  Test shape: (3576, 3217)
Train fraud rate: 4.84%
Test fraud rate:  4.84%


## Part I: Save Artifacts

Everything downstream (model training, SHAP explanations, the Streamlit app)
needs to reuse the **exact same** fitted `TfidfVectorizer` and `OneHotEncoder`
— never refit them on new data, or the feature space at inference time won't
match the one the models were trained on.

In [11]:
os.makedirs("../data/processed", exist_ok=True)

with open("../data/processed/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("../data/processed/onehot_encoder.pkl", "wb") as f:
    pickle.dump(ohe, f)

with open("../data/processed/train_test_split.pkl", "wb") as f:
    pickle.dump({
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "suspicious_phrases": SUSPICIOUS_PHRASES
    }, f)

print("Artifacts saved to ../data/processed/ ✅")

Artifacts saved to ../data/processed/ ✅


## Summary

| Step | Output |
|---|---|
| Missing value handling | 0 nulls remaining; 2 high-missingness columns converted to binary flags instead of dropped |
| Text cleaning | `full_text` column: lowercased, HTML/URL/email/punctuation/digit-stripped |
| Feature engineering | 11 suspicious-phrase flags + 3 aggregate signals (`suspicious_phrase_count`, `description_length`, `has_email_gmail`) |
| TF-IDF | Up to 3000 features, unigrams + bigrams |
| Categorical encoding | One-hot, unknown-safe for inference |
| Final feature matrix | TF-IDF + categorical + numeric blocks combined via sparse hstack |
| Train/test split | 80/20, stratified on `fraudulent` to preserve class imbalance ratio |

**Next step:** `03_Model_Development.ipynb` — train Logistic Regression, Naive
Bayes, SVM, Random Forest, and XGBoost on `X_train`/`y_train`, compare using
Accuracy, Precision, Recall, F1, and ROC-AUC on `X_test`/`y_test`.